In [ ]:
# Import the modules
import os
import pandas as pd
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-bright')
from scipy.stats import linregress

# Unzip folder
zipped_path = r"D:\Downloads\Metadata 188405.zip"
unzipped_path = r"D:\meta data unzipped"

with zipfile.ZipFile(zipped_path, 'r') as zip_ref:
    zip_ref.extractall(unzipped_path)

# Read the csv files
df_sample1 = pd.read_csv(os.path.join(unzipped_path, "chromosomal-abnormalities.csv"))
df_sample2 = pd.read_csv(os.path.join(unzipped_path, "decedents.csv"))
df_sample3 = pd.read_csv(os.path.join(unzipped_path, "medical-history.csv"))

# Merge csv files
merge_cols = ["id", "deidentified_record_number"]

df_master = df_sample1.merge(df_sample2,
    on=merge_cols,
    how='outer',
).merge(df_sample3,
    on=merge_cols,
    how='outer'
)

# Create keywords
keywords = 'hip|pelvis|femur|leg'

# Find rows containing keywords
mask = df_master.astype(str).apply(lambda x: x.str.contains(keywords, case=False, na=False))
rows_with_keywords = mask.any(axis=1)

# Define all_keyword_rows
all_keyword_rows = df_master[rows_with_keywords]

# Calculate keyword totals 
keyword_totals = {
    'Hip': all_keyword_rows.apply(lambda row: row.astype(str).str.contains('hip', case=False).any(), axis=1).sum(),
    'Pelvis': all_keyword_rows.apply(lambda row: row.astype(str).str.contains('pelvis', case=False).any(), axis=1).sum(),
    'Femur': all_keyword_rows.apply(lambda row: row.astype(str).str.contains('femur', case=False).any(), axis=1).sum(),
    'Leg': all_keyword_rows.apply(lambda row: row.astype(str).str.contains('leg', case=False).any(), axis=1).sum()
}

# Output 1: all rows with keywords including multiple per number
all_keyword_rows.to_csv(os.path.join(unzipped_path, "all_keyword_rows.csv"), index=False)

# Output 2: just one patient number
unique_patient_numbers = pd.DataFrame(all_keyword_rows['deidentified_record_number'].unique(), columns=['deidentified_record_number'])
unique_patient_numbers.to_csv(os.path.join(unzipped_path, "unique_patient_numbers.csv"), index=False)

# Pie chart
plt.figure(figsize=(7,7))
plt.pie(keyword_totals.values(), labels=keyword_totals.keys(), autopct='%1.1f%%')
plt.title('Keyword Distribution')
plt.tight_layout()
plt.savefig(os.path.join(unzipped_path, 'keyword_pie.png'))
plt.close()

# Age plots
if 'age_months' in all_keyword_rows.columns:
    # Histogram
    plt.figure(figsize=(10,5))
    plt.hist(all_keyword_rows['age_months'], bins=20)
    plt.title('Age Distribution (Months)')
    plt.xlabel('Age (Months)')
    plt.ylabel('Records')
    plt.xticks(range(0, int(max(all_keyword_rows['age_months']))+12, 12))
    plt.tight_layout()
    plt.savefig(os.path.join(unzipped_path, 'age_histogram.png'))
    plt.close()

    # Violin plot
    plt.figure(figsize=(12, 6))

    # Prepare data for seaborn
    plot_data = []
    for keyword in ['hip', 'pelvis', 'femur', 'leg']:
        mask = all_keyword_rows.apply(lambda row: row.astype(str).str.contains(keyword, case=False).any(), axis=1)
        temp_df = all_keyword_rows.loc[mask, ['age_months']].copy()
        temp_df['Keyword'] = keyword.capitalize()
        plot_data.append(temp_df)

    plot_df = pd.concat(plot_data)

    hue_palette = {
        'Hip': '#d62728',    
        'Pelvis': '#ff7f0e',  
        'Femur': "#e56dc1",   
        'Leg': '#17becf'      

    }

    # Create violin plot
    sns.violinplot(
        data=plot_df,
        x='Keyword',
        y='age_months',
        hue='Keyword',
        palette=hue_palette,
        inner='quartile',
        cut=0
    )

    # Style enhancements
    plt.title('Age Distribution by Keyword Type (Months)', pad=20, fontsize=14)
    plt.xlabel('Keyword', labelpad=10)
    plt.ylabel('Age (Months)', labelpad=10)
    plt.grid(axis='y', alpha=0.3)

    # Add data point count annotations
    for i, keyword in enumerate(['Hip', 'Pelvis', 'Femur', 'Leg']):
        count = len(plot_df[plot_df['Keyword'] == keyword])
        plt.text(i, plot_df['age_months'].max()*1.05, 
                f'n={count}', 
                ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(unzipped_path, 'age_keyword_violinplot.png'), dpi=300)
    plt.close()
